# Chunking + Embedding

같은 디렉토리의 `input.csv`를 읽어 `content`를 청킹하고, OpenAI embedding vector를 `float32` BLOB(Base64 문자열)으로 변환해 `result.csv`로 저장합니다.

- 입력 컬럼: `content`, `feature`
- 출력 컬럼: 원본 컬럼 + `index` + `blob`
- `index`: 같은 `feature` 내부에서 청크 순서대로 1부터 부여

In [ ]:
import base64
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

In [ ]:
# 1. 기본 설정
BASE_DIR = Path.cwd()
INPUT_CSV = BASE_DIR / "input.csv"
OUTPUT_CSV = BASE_DIR / "result.csv"

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_BATCH_SIZE = 100

# 문자 기준 청킹 설정입니다. 데이터 성격에 맞게 조정하세요.
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 120

load_dotenv(BASE_DIR / ".env")
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY가 .env에 없습니다.")

client = OpenAI(api_key=api_key)

In [ ]:
# 2. input.csv 로드 및 필수 컬럼 확인
df = pd.read_csv(INPUT_CSV)

required_columns = {"content", "feature"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"input.csv에 필수 컬럼이 없습니다: {sorted(missing_columns)}")

print(f"원본 데이터 개수: {len(df)}개")

df = df.dropna(subset=["content", "feature"]).copy()
df["content"] = df["content"].astype(str)
df["feature"] = df["feature"].astype(str)
df = df[df["content"].str.strip().str.len() > 0].reset_index(drop=True)

print(f"전처리 후 데이터 개수: {len(df)}개")

In [ ]:
# 3. 청킹 함수
def normalize_text(text: str) -> str:
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    text = normalize_text(text)
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    min_break = int(chunk_size * 0.6)

    while start < len(text):
        end = min(start + chunk_size, len(text))
        window = text[start:end]

        if end < len(text):
            break_points = [window.rfind(sep) for sep in ["\n\n", "\n", ". ", "? ", "! ", "。", " "]]
            best_break = max(break_points)
            if best_break >= min_break:
                end = start + best_break + 1
                window = text[start:end]

        chunk = window.strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break
        start = max(end - overlap, start + 1)

    return chunks

In [ ]:
# 4. content를 청크 행으로 확장하고 feature별 index 부여
rows = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="청킹 중"):
    chunks = split_text(row["content"])
    for chunk in chunks:
        item = row.to_dict()
        item["content"] = chunk
        rows.append(item)

chunked_df = pd.DataFrame(rows, columns=df.columns)
chunked_df["index"] = chunked_df.groupby("feature").cumcount() + 1

print(f"청킹 후 데이터 개수: {len(chunked_df)}개")
chunked_df.head()

In [ ]:
# 5. 임베딩 및 BLOB 변환 함수
def embedding_to_blob(embedding: list[float]) -> str:
    vec = np.array(embedding, dtype=np.float32)
    return base64.b64encode(vec.tobytes()).decode("utf-8")


def get_blob_embeddings(text_list: list[str], batch_size: int = EMBEDDING_BATCH_SIZE) -> list[str]:
    all_blobs = []

    for i in tqdm(range(0, len(text_list), batch_size), desc="임베딩 변환 중"):
        batch = text_list[i : i + batch_size]
        response = client.embeddings.create(
            input=batch,
            model=EMBEDDING_MODEL,
        )
        all_blobs.extend(embedding_to_blob(data.embedding) for data in response.data)

    return all_blobs

In [ ]:
# 6. 실행 및 result.csv 저장
texts = chunked_df["content"].fillna("").astype(str).tolist()
chunked_df["blob"] = get_blob_embeddings(texts)

# 원본 컬럼 순서를 유지하고 index, blob을 맨 뒤에 둡니다.
output_columns = list(df.columns) + ["index", "blob"]
chunked_df = chunked_df[output_columns]
chunked_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_CSV}")
chunked_df.head()